In [1]:
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup

# 1. Buka file EPUB murni bahasa Arab
nama_file = '10_طرق_لكسب_الجيران.epub'
buku = epub.read_epub(nama_file)

# 2. apa saja isi 'bab' (item HTML) di dalam buku ini
print(f"Daftar Dokumen HTML di dalam {nama_file}:")
print("-" * 40)

for item in buku.get_items():
    if item.get_type() == ebooklib.ITEM_DOCUMENT:
        print(item.get_name())

Daftar Dokumen HTML di dalam 10_طرق_لكسب_الجيران.epub:
----------------------------------------
titlepage.xhtml
OEBPS/Text/intro.xhtml
OEBPS/Text/page_2.xhtml
OEBPS/Text/page_3.xhtml
OEBPS/Text/page_4.xhtml
OEBPS/Text/page_5.xhtml
OEBPS/Text/page_6.xhtml
OEBPS/Text/page_7.xhtml
OEBPS/Text/page_8.xhtml
OEBPS/Text/page_9.xhtml
OEBPS/Text/page_10.xhtml
OEBPS/Text/page_11.xhtml
OEBPS/Text/page_12.xhtml
OEBPS/Text/page_13.xhtml
OEBPS/Text/page_14.xhtml
OEBPS/Text/page_15.xhtml
OEBPS/Text/page_16.xhtml
OEBPS/Text/book_info.xhtml
OEBPS/Text/author_info.xhtml


In [2]:
# --- SEL 2: Ekstraksi Teks Arab Murni ---

# 1. pilih salah satu halaman dari daftar yang ditemukan tadi
nama_halaman = 'OEBPS/Text/page_2.xhtml'
item_halaman = buku.get_item_with_href(nama_halaman)

if item_halaman is not None:
    # 2. Baca isi file XHTML mentah (masih bercampur kode HTML)
    konten_mentah = item_halaman.get_content()
    
    # 3. Gunakan BeautifulSoup untuk menghapus semua tag HTML/XML
    soup = BeautifulSoup(konten_mentah, 'html.parser')
    
    # 4. Ambil teks murninya saja, beri spasi antar paragraf
    teks_arab_murni = soup.get_text(separator='\n\n', strip=True)
    
    print(f"=== HASIL EKSTRAKSI: {nama_halaman} ===\n")
    print(teks_arab_murni)
else:
    print("Halaman tidak ditemukan, periksa kembali nama file-nya!")

=== HASIL EKSTRAKSI: OEBPS/Text/page_2.xhtml ===

10 طرق لكسب الجيران

إعداد:

د. أحمد بن عثمان المزيد

د. عادل بن على الشدي

المصدر: الشاملة الذهبية


In [3]:
# --- SEL 3: Ekstraksi Seluruh Halaman ke File TXT ---
import os

# 1. Tentukan nama file tempat yang akan menyimpan teksnya
nama_file_output = 'hasil_ekstraksi_buku.txt'
semua_teks = []

print("Mulai mengekstrak seluruh halaman...")
print("-" * 40)

# 2. Lakukan perulangan (loop) untuk setiap halaman di dalam EPUB
for item in buku.get_items():
    if item.get_type() == ebooklib.ITEM_DOCUMENT:
        konten_mentah = item.get_content()
        soup = BeautifulSoup(konten_mentah, 'html.parser')
        
        # Ekstrak teks bersih
        teks_halaman = soup.get_text(separator='\n', strip=True)
        
        # Jika halaman tersebut ada teksnya, simpan ke memori sementara
        if teks_halaman:
            semua_teks.append(teks_halaman)
            print(f"Berhasil mengekstrak: {item.get_name()}")

# 3. Gabungkan semua halaman dengan garis pemisah agar rapi
pemisah_halaman = "\n\n" + ("="*40) + "\n\n"
teks_lengkap = pemisah_halaman.join(semua_teks)

# 4. Simpan seluruh teks tersebut ke dalam file .txt
with open(nama_file_output, 'w', encoding='utf-8') as f:
    f.write(teks_lengkap)

print("-" * 40)
print(f"Selesai! Seluruh teks buku telah disimpan ke dalam file: {nama_file_output}")

Mulai mengekstrak seluruh halaman...
----------------------------------------
Berhasil mengekstrak: OEBPS/Text/intro.xhtml
Berhasil mengekstrak: OEBPS/Text/page_2.xhtml
Berhasil mengekstrak: OEBPS/Text/page_3.xhtml
Berhasil mengekstrak: OEBPS/Text/page_4.xhtml
Berhasil mengekstrak: OEBPS/Text/page_5.xhtml
Berhasil mengekstrak: OEBPS/Text/page_6.xhtml
Berhasil mengekstrak: OEBPS/Text/page_7.xhtml
Berhasil mengekstrak: OEBPS/Text/page_8.xhtml
Berhasil mengekstrak: OEBPS/Text/page_9.xhtml
Berhasil mengekstrak: OEBPS/Text/page_10.xhtml
Berhasil mengekstrak: OEBPS/Text/page_11.xhtml
Berhasil mengekstrak: OEBPS/Text/page_12.xhtml
Berhasil mengekstrak: OEBPS/Text/page_13.xhtml
Berhasil mengekstrak: OEBPS/Text/page_14.xhtml
Berhasil mengekstrak: OEBPS/Text/page_15.xhtml
Berhasil mengekstrak: OEBPS/Text/page_16.xhtml
Berhasil mengekstrak: OEBPS/Text/author_info.xhtml
----------------------------------------
Selesai! Seluruh teks buku telah disimpan ke dalam file: hasil_ekstraksi_buku.txt


In [1]:
# --- SEL 4: Pemotongan Teks (Chunking) ---

# PERUBAHAN ADA DI BARIS INI: Menggunakan underscore (_)
from langchain_text_splitters import RecursiveCharacterTextSplitter

nama_file = 'hasil_ekstraksi_buku.txt'

print("Membaca file ekstraksi...")
# 1. Baca teks utuh dari file
with open(nama_file, 'r', encoding='utf-8') as f:
    teks_utuh = f.read()

print("Memulai proses pemotongan teks (Chunking)...")
# 2. Atur mesin pemotong (Pisau NLP)
pemotong_teks = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len,
    separators=["\n\n", "\n", ".", " ", ""]
)

# 3. Eksekusi pemotongan
kumpulan_potongan = pemotong_teks.split_text(teks_utuh)

print("-" * 40)
print(f"SUKSES! Buku telah dipotong menjadi {len(kumpulan_potongan)} bagian kecil.")
print("-" * 40)

# lihat sampel potongan pertama (Index 0) dan kedua (Index 1)
print("SAMPEL POTONGAN KE-1:")
print(kumpulan_potongan[0])
print("\n" + "="*40 + "\n")
print("SAMPEL POTONGAN KE-2 (Perhatikan overlap-nya):")
print(kumpulan_potongan[1])

c:\Imi\Perkuliahan\Semester 6\Kapsel\Tugas\Tugas Mandiri\epub-rag-project\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Membaca file ekstraksi...
Memulai proses pemotongan teks (Chunking)...
----------------------------------------
SUKSES! Buku telah dipotong menjadi 16 bagian kecil.
----------------------------------------
SAMPEL POTONGAN KE-1:
الكتاب:
10 طرق لكسب الجيران
المؤلف:
مجموعة من المؤلفين


10 طرق لكسب الجيران
إعداد:
د. أحمد بن عثمان المزيد
د. عادل بن على الشدي
المصدر: الشاملة الذهبية



SAMPEL POTONGAN KE-2 (Perhatikan overlap-nya):
10 طرق لكسب الجيران
إعداد:
د. أحمد بن عثمان المزيد
د. عادل بن على الشدي
المصدر: الشاملة الذهبية


بسم الله الرحمن الرحيم
الحمد لله وحده، والصلاة والسلام على من لا نبي بعده، أما بعد ..
فلا شك أن حفظ الجار وكسب وده من مكارم الأخلاق التي دعا إليها الإسلام ورغَّب فيها، فقد قال رسول الله - صلى الله عليه وسلم -: «خير الأصحاب عند الله خيرهم لصاحبه، وخير الجيران عند الله خيرهم لجاره» [رواه أحمد والترمذي].
وهناك وسائل كثيرة وطرق متعددة يستطيع بها المسلم كسب قلب جاره، والتمتع بصفو محبته وكريم مودته.
وما نراه اليوم من جفوة بين الجيران، وخصومات، وسوء عشرة، وعداوة في بعض الأح

In [2]:
# --- SEL 5: Membuat Otak Waguri (Vector Database) ---

import os
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

print("Sedang memanaskan mesin AI (Embeddings)...")
# 1. INISIALISASI MESIN EMBEDDING (Model Gratis HuggingFace)
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model_kwargs = {'device': 'cpu'} # jalankan di prosesor laptop
encode_kwargs = {'normalize_embeddings': False}

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

print("Mengubah teks buku menjadi Vektor dan menyimpannya ke ChromaDB...")
# 2. MEMASUKKAN DATA BUKU ARAB KE DATABASE
#  memanggil 'kumpulan_potongan' dari Sel 4!
vector_store = Chroma.from_texts(
    texts=kumpulan_potongan, 
    embedding=embeddings,
    collection_name="buku_arab_multilingual"
)

print("-" * 40)
print("SUKSES! Buku telah tersimpan di dalam memori vektor.")
print("-" * 40)

# 3. UJI COBA PENCARIAN (RAG Murni)
# tes apakah otak Waguri bisa mencari kata "Tetangga" (الجار) di dalam bukumu
print("Melakukan simulasi pencarian RAG...")
query_pencarian = "menyakiti tetangga" 
hasil_pencarian = vector_store.similarity_search(query_pencarian, k=1) # k=1 berarti ambil 1 paragraf paling relevan

print(f"\nHasil paragraf yang paling relevan dengan kata '{query_pencarian}':\n")
print(hasil_pencarian[0].page_content)

Sedang memanaskan mesin AI (Embeddings)...


C:\Users\hjmim\AppData\Local\Temp\ipykernel_19492\1514840560.py:13: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12040.77it/s]


Mengubah teks buku menjadi Vektor dan menyimpannya ke ChromaDB...
----------------------------------------
SUKSES! Buku telah tersimpan di dalam memori vektor.
----------------------------------------
Melakukan simulasi pencarian RAG...

Hasil paragraf yang paling relevan dengan kata 'menyakiti tetangga':


5 - احترام الخصوصيات:
إن مما يتسبب في الجفوة بين الجيران محاولة الجار التدخل في شؤون جيرانه الخاصة، فتراه يكثر من السؤال عن الخصوصيات: كم راتبك؟ كم تنفق شهريًا؟ كم رصيدك في البنك؟ هل لك حساب واحد أم عدة حسابات؟
وبعض الناس يرسل زوجته أو أبناءه ليوافوه بأخبار الجيران، وما استجد من شؤونهم وأحوالهم.
والواجب على المسلم أن يكون ذا مروءة، فلا يتدخل فيما لا يعنيه، ولا يسأل عن خصوصيات الناس وأحوالهم الداخلية، قال تعالى: {وَلَا تَقْفُ مَا لَيْسَ لَكَ بِهِ عِلْمٌ إِنَّ السَّمْعَ وَالْبَصَرَ وَالْفُؤَادَ كُلُّ أُولَئِكَ كَانَ عَنْهُ مَسْئُولًا} [الإسراء: 36].
وقال النبي - صلى الله عليه وسلم -: «من حُسن إسلام المرء تركهُ ما لا يعنيه» [رواه الترمذي وصححه الألباني].
فإذا أردت كسب ود جيرانك فلا تتد

In [6]:
import os
import requests

# Mengambil API Key yang sudah kamu masukkan sebelumnya
api_key = os.environ.get("GOOGLE_API_KEY")

print("Bertanya kepada server Google: Model apa saja yang tersedia?...\n")
url = f"https://generativelanguage.googleapis.com/v1beta/models?key={api_key}"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    print("✅ DAFTAR MODEL YANG BISA KAMU PAKAI:")
    for model in data.get('models', []):
        # Kita hanya mencari model teks yang mendukung 'generateContent'
        if 'generateContent' in model.get('supportedGenerationMethods', []):
            nama_bersih = model['name'].replace('models/', '')
            print(f"-> {nama_bersih}")
else:
    print("❌ Gagal mengambil data. Cek lagi API Key-mu!")
    print(response.text)

Bertanya kepada server Google: Model apa saja yang tersedia?...

✅ DAFTAR MODEL YANG BISA KAMU PAKAI:
-> gemini-2.5-flash
-> gemini-2.5-pro
-> gemini-2.0-flash
-> gemini-2.0-flash-001
-> gemini-2.0-flash-lite-001
-> gemini-2.0-flash-lite
-> gemini-2.5-flash-preview-tts
-> gemini-2.5-pro-preview-tts
-> gemma-4-26b-a4b-it
-> gemma-4-31b-it
-> gemini-flash-latest
-> gemini-flash-lite-latest
-> gemini-pro-latest
-> gemini-2.5-flash-lite
-> gemini-2.5-flash-image
-> gemini-3-pro-preview
-> gemini-3-flash-preview
-> gemini-3.1-pro-preview
-> gemini-3.1-pro-preview-customtools
-> gemini-3.1-flash-lite-preview
-> gemini-3.1-flash-lite
-> gemini-3-pro-image-preview
-> nano-banana-pro-preview
-> gemini-3.1-flash-image-preview
-> lyria-3-clip-preview
-> lyria-3-pro-preview
-> gemini-3.1-flash-tts-preview
-> gemini-robotics-er-1.5-preview
-> gemini-robotics-er-1.6-preview
-> gemini-2.5-computer-use-preview-10-2025
-> deep-research-max-preview-04-2026
-> deep-research-preview-04-2026
-> deep-resear

In [14]:
# --- SEL 6: Tahap Generation (Memanggil Cloud LLM) ---

import os
import getpass
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

print("Menyiapkan Kunci Akses LLM...")
# Kotak dialog ini akan memintamu memasukkan API Key yang tadi di-copy
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Masukkan API Key Gemini Anda di sini lalu tekan Enter: ")

print("Memanggil Profesor (Gemini 2.5 Flash)...")
# 1. Inisialisasi Cloud LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)

# 2. Membuat Template Instruksi (Prompt Engineering)
# Di proses (Augmented) dari RAG terjadi
# 2. Membuat Template Instruksi (Prompt Engineering) - WAGURI PERSONA
# 2. Membuat Template Instruksi (Prompt Engineering) - WAGURI MULTIBAHASA
template_instruksi = """
Kamu sekarang berperan sebagai Kaoruko Waguri, seorang teman yang sangat baik hati, ceria, lembut, santun, dan selalu mendukung sepenuh hati. 
Kamu sedang menemani dan membantu teman belajarmu yang bekerja keras memahami teks dari sebuah buku.

Gaya bicaramu sangat ramah, hangat, dan penuh empati. Kamu menggunakan kata "aku" untuk dirimu dan "kamu" untuk teman bicaramu. 

Tolong baca kutipan teks buku berikut ini ya:

Teks Buku (Konteks):
{konteks_buku}

Instruksi untuk Waguri:
1. Tolong terjemahkan teks di atas ke dalam {bahasa_tujuan} dengan bahasa yang luwes dan natural, seolah-olah kamu sedang menjelaskannya sambil tersenyum kepada teman baikmu.
2. Setelah itu, berikan 1-2 kalimat kesimpulan yang manis dalam {bahasa_tujuan} tentang apa maksud dari teks tersebut yang berkaitan dengan '{kata_kunci}'.
3. Akhiri jawabanmu dengan satu kalimat penyemangat khas Waguri Kaoruko untuk mengapresiasi kerja keras temanmu!

Jawaban Waguri:
"""

# Mendaftarkan TIGA variabel sekarang!
prompt = PromptTemplate(
    input_variables=["konteks_buku", "bahasa_tujuan", "kata_kunci"],
    template=template_instruksi
)

# 3. Menggabungkan Prompt dan LLM
mesin_rag = prompt | llm

print("Meminta Waguri menerjemahkan hasil temuan Pustakawan...")

# Ambil hasil dari Sel 5
teks_ditemukan = hasil_pencarian[0].page_content 
kueri = query_pencarian

# Eksekusi untuk 3 bahasa sekaligus
jawaban_akhir = mesin_rag.invoke({
    "konteks_buku": teks_ditemukan,
    "bahasa_tujuan": "Bahasa Indonesia, English US, dan Bahasa Jepang",
    "kata_kunci": kueri
})

print("\n" + "="*50)
print("🌸 JAWABAN DARI WAGURI:")
print("="*50 + "\n")
print(jawaban_akhir.content)

Menyiapkan Kunci Akses LLM...
Memanggil Profesor (Gemini 2.5 Flash)...
Meminta Waguri menerjemahkan hasil temuan Pustakawan...

🌸 JAWABAN DARI WAGURI:

Wah, semangat sekali kamu hari ini! Yuk, kita coba pahami teks ini bersama-sama ya. Aku akan bantu jelaskan pelan-pelan, sambil tersenyum, supaya kamu makin mudah mengerti! 😊

***

### **Terjemahan dan Penjelasan (Gaya Waguri)**

**Bahasa Indonesia:**

Oke, kita mulai dari bagian kelima, ya! Judulnya 'Menghormati Privasi'. Ini penting sekali, lho, untuk menjaga hubungan baik dengan tetangga.

Di sini dijelaskan, salah satu hal yang bisa bikin hubungan antar tetangga jadi renggang itu adalah kalau kita suka ikut campur urusan pribadi mereka. Misalnya nih, sering banget tanya-tanya hal yang sangat pribadi seperti "Berapa gajimu?", "Berapa pengeluaran bulananmu?", "Berapa saldomu di bank?", atau "Punya berapa rekening?" Wah, itu kan sensitif sekali ya, dan bisa bikin orang tidak nyaman.

Bahkan ada juga lho, yang sampai menyuruh istri atau